# LUMEN Data Science: Student Performance Analysis

## 1. Problem Statement & Business Questions
Tujuan proyek ini adalah untuk menganalisis performa siswa di berbagai tingkatan (Beginner, Intermediate, Advanced) dan mengidentifikasi faktor utama yang menentukan kelulusan mereka. Kami juga akan menguji apakah fitur 'Rekomendasi Level' memiliki dampak positif pada rata-rata nilai kuis siswa.

**Pertanyaan Bisnis:**
1. Bagaimana distribusi skor siswa di tiap tingkatan?
2. Apakah siswa yang menggunakan 'Fitur Rekomendasi' mendapatkan nilai yang lebih tinggi secara signifikan?
3. Fitur apa yang paling memengaruhi kemungkinan seorang siswa lulus (mendapat nilai >= 75)?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

## 2. Data Wrangling (Gathering, Assessing, Cleaning)

**Gathering Data**: Kita akan memuat data mentah `raw_student_data.csv` yang disimulasikan dari log backend.

In [ ]:
df_raw = pd.read_csv('raw_student_data.csv')
print("Shape data mentah:", df_raw.shape)
df_raw.head()

### 2.1. Assessing Data
Mengevaluasi struktur dan kualitas data, termasuk mengecek nilai *missing values* dan data duplikat.

In [ ]:
print("Missing Values:\n", df_raw.isnull().sum())
print("\nJumlah Data Duplikat:", df_raw.duplicated().sum())

### 2.2. Cleaning Data
Kita akan melakukan **pembersihan secara manual**:
1. Menghapus data yang benar-benar terduplikat.
2. Mengisi nilai yang kosong (*missing values*) pada kolom `quiz_score` dengan median dari grup level yang sesuai, agar tidak merusak distribusi.

In [ ]:
# Drop duplikat
df_clean = df_raw.drop_duplicates()

# Isi missing value skor dengan nilai median grup levelnya
df_clean['quiz_score'] = df_clean.groupby('current_level')['quiz_score'].transform(lambda x: x.fillna(x.median()))

# Isi missing value time_spent dengan median keseluruhan
df_clean['time_spent_mins'] = df_clean['time_spent_mins'].fillna(df_clean['time_spent_mins'].median())

print("Missing Values setelah cleaning:\n", df_clean.isnull().sum())
print("Shape data bersih:", df_clean.shape)

## 3. Exploratory Data Analysis (EDA)

**Q1. Distribusi Skor per Level**
Mari kita visualisasikan bagaimana perbedaan skor antara Beginner, Intermediate, dan Advanced.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x='current_level', y='quiz_score', order=['Beginner', 'Intermediate', 'Advanced'])
plt.title('Distribusi Skor Berdasarkan Level')
plt.xlabel('Tingkatan (Level)')
plt.ylabel('Skor Kuis')
plt.savefig('eda_boxplot.png') # Menyimpan grafik untuk dashboard
plt.show()

**Explanatory Analysis (Kesimpulan Grafik):** Berdasarkan grafik *box-plot* di atas, siswa tingkat *Beginner* cenderung memiliki skor rata-rata yang lebih tinggi dengan variasi yang lebih sempit dibandingkan *Advanced*. Siswa *Advanced* menghadapi materi yang lebih menantang sehingga persebaran skornya lebih luas.

## 4. A/B Testing Simulation
**Q2. Apakah siswa yang menggunakan 'Fitur Rekomendasi' mendapatkan nilai yang lebih tinggi secara signifikan?**

Kita menguji efektivitas `Fitur Rekomendasi Level` (Grup B) berbanding tanpa fitur (Grup A).
- H0: Fitur rekomendasi **tidak** memberikan perbedaan signifikan pada rata-rata skor kuis.
- H1: Fitur rekomendasi memberikan **peningkatan** signifikan pada rata-rata skor kuis.

In [ ]:
group_a = df_clean[df_clean['ab_test_group'] == 'Group A']['quiz_score']
group_b = df_clean[df_clean['ab_test_group'] == 'Group B']['quiz_score']

print(f"Rata-rata Grup A (Tanpa Fitur): {group_a.mean():.2f}")
print(f"Rata-rata Grup B (Dengan Fitur) : {group_b.mean():.2f}")

# Menggunakan Independent T-Test (One-Sided/Greater)
t_stat, p_val = stats.ttest_ind(group_b, group_a, alternative='greater')
print(f"\nT-Statistic: {t_stat:.2f}, P-Value: {p_val:.4f}")

if p_val < 0.05:
    print("\nKesimpulan: P-Value < 0.05. Tolak H0.\nFitur Rekomendasi terbukti MENINGKATKAN skor siswa secara signifikan.")
else:
    print("\nKesimpulan: P-Value >= 0.05. Gagal tolak H0.\nBelum cukup bukti bahwa fitur rekomendasi meningkatkan skor.")

## 5. Feature Engineering & Machine Learning
**Q3. Fitur apa yang paling memengaruhi kemungkinan seorang siswa lulus (mendapat nilai >= 75)?**

Kita akan melakukan **Feature Engineering** dengan membuat target baru: `is_passed` (1 jika Lulus, 0 jika Gagal).
Untuk menghindari **Data Leakage**, kita harus menghapus (*drop*) kolom `quiz_score` (Target mentah) dari fitur yang akan dilatih (*X*).

In [ ]:
# Feature Engineering: Menentukan Target Biner
df_clean['is_passed'] = (df_clean['quiz_score'] >= 75).astype(int)

# Menyimpan dataset yang bersih dan sudah memiliki target biner
df_clean.to_csv('clean_student_data.csv', index=False)
print("Data bersih disimpan ke 'clean_student_data.csv'")

# --- MACHINE LEARNING MODELING ---
df_model = df_clean.copy()
# One-Hot Encoding untuk variabel kategorikal
df_model = pd.get_dummies(df_model, columns=['current_level', 'ab_test_group'], drop_first=True)

# Pencegahan Data Leakage (Kolom 'quiz_score' dan 'is_passed' dihapus dari X)
X = df_model.drop(columns=['student_id', 'quiz_score', 'is_passed'])
y = df_model['is_passed']

# Split Train/Test Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Training Model (Decision Tree)
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

# Evaluasi
y_pred = model.predict(X_test)
print("\nAkurasi Model:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

### Fitur yang Paling Mempengaruhi Kelulusan

In [ ]:
# Menampilkan Feature Importance
importance = pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
importance = importance.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=importance, x='Importance', y='Feature')
plt.title('Feature Importance (Faktor Kelulusan)')
plt.show()